## NB_hospital_daily_summary
Builds `<catalog>.gold.hospital_daily_summary` from the silver layer, following STTM `sttm_silver_to_gold.xlsx` (sheet: hospital_daily_summary).

* **Grain:** one row per hospital per calendar day (days with lab activity)
* **Driving table:** `silver.lab_results` grouped by hospital_id, test_date
* **Joins:** LEFT JOIN `silver.hospitals` on hospital_id; LEFT JOIN `silver.calendar` on date_key = yyyyMMdd(test_date)
* **Window:** LAG(total_tests) partitioned by hospital_id ordered by summary_date (previous day with activity)
* **Load type:** FULL - `DataFrame.writeTo(...).createOrReplace()` (PySpark equivalent of CREATE OR REPLACE TABLE ... AS SELECT)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "dbw_caresync")
catalog = dbutils.widgets.get("catalog").strip() or "dbw_caresync"

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.hospital_daily_summary"
print(f"[SOURCE SCHEMA] {source_schema}")
print(f"[TARGET TABLE] {target_table}")

In [0]:
print("[STEP 1] Reading silver sources")
lab_results_df = spark.table(f"{source_schema}.lab_results")
hospitals_df = spark.table(f"{source_schema}.hospitals").alias("h")
calendar_df = spark.table(f"{source_schema}.calendar").alias("cal")

# Exact decimal 100.0 (same as the SQL literal) so rounding of the rate columns is deterministic
HUNDRED = F.lit(100.0).cast("decimal(4,1)")


def status_flag(status):
    """1 when result_status equals the given value, else 0 (used for conditional counts)."""
    return F.when(F.col("result_status") == status, 1).otherwise(0)


print("[STEP 2] Aggregating lab_results per hospital per day")
daily_df = lab_results_df.groupBy("hospital_id", F.col("test_date").alias("summary_date")).agg(
    F.count("*").alias("total_tests"),
    F.countDistinct("patient_id").alias("unique_patients_tested"),
    F.countDistinct("doctor_id").alias("active_doctor_count"),
    F.sum(status_flag("Abnormal")).alias("abnormal_test_count"),
    F.sum(status_flag("Critical")).alias("critical_test_count"),
)

print("[STEP 3] Day-over-day trend: LAG(total_tests) per hospital ordered by summary_date")
hospital_window = Window.partitionBy("hospital_id").orderBy("summary_date")
with_lag_df = daily_df.withColumn(
    "prev_day_total_tests", F.lag("total_tests").over(hospital_window)
).alias("w")

print("[STEP 4] Joining hospitals + calendar (LEFT) and applying column mapping")
prev = F.col("w.prev_day_total_tests")
hospital_daily_summary_df = (
    with_lag_df
    .join(hospitals_df, F.col("w.hospital_id") == F.col("h.hospital_id"), "left")
    .join(
        calendar_df,
        F.col("cal.date_key") == F.date_format(F.col("w.summary_date"), "yyyyMMdd").cast("int"),
        "left",
    )
    .select(
        F.col("w.hospital_id").cast("int").alias("hospital_id"),
        F.col("h.hospital_name").cast("string").alias("hospital_name"),
        F.col("w.summary_date").cast("date").alias("summary_date"),
        F.col("cal.day_name").cast("string").alias("day_name"),
        F.col("cal.is_weekend").cast("boolean").alias("is_weekend"),
        F.col("w.total_tests").cast("int").alias("total_tests"),
        F.col("w.unique_patients_tested").cast("int").alias("unique_patients_tested"),
        F.col("w.active_doctor_count").cast("int").alias("active_doctor_count"),
        F.col("w.abnormal_test_count").cast("int").alias("abnormal_test_count"),
        F.col("w.critical_test_count").cast("int").alias("critical_test_count"),
        (F.col("w.abnormal_test_count") * HUNDRED / F.col("w.total_tests")).cast("decimal(5,2)").alias("abnormal_rate_pct"),
        (F.col("w.critical_test_count") * HUNDRED / F.col("w.total_tests")).cast("decimal(5,2)").alias("critical_rate_pct"),
        prev.cast("int").alias("prev_day_total_tests"),
        F.when(prev.isNull() | (prev == 0), F.lit(None))
        .otherwise((F.col("w.total_tests") - prev) * HUNDRED / prev)
        .cast("decimal(6,2)")
        .alias("test_volume_change_pct"),
    )
)

print(f"[STEP 5] Writing {target_table} (full rebuild)")
hospital_daily_summary_df.writeTo(target_table).using("delta").createOrReplace()
spark.sql(
    f"COMMENT ON TABLE {target_table} IS "
    "'Gold: one row per hospital per day with lab volume, abnormal/critical rates and day-over-day trend. Built per STTM silver-to-gold v1.0.'"
)
print(f"[STEP 5 COMPLETE] {target_table} rebuilt")

In [0]:
print("[STEP 6] Validating grain and reconciliation")
stats = (
    spark.table(target_table)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("hospital_id", "summary_date").alias("key_count"),
        F.sum("total_tests").alias("test_count"),
    )
    .first()
)
src_tests = spark.table(f"{source_schema}.lab_results").count()
print(f"[TARGET ROWS] {stats.row_count} | [DISTINCT KEYS] {stats.key_count} | [SUM total_tests] {stats.test_count} | [SOURCE lab_results] {src_tests}")

if stats.row_count != stats.key_count:
    raise ValueError("Grain violation: duplicate (hospital_id, summary_date)")
if stats.test_count != src_tests:
    raise ValueError("Reconciliation failed: SUM(total_tests) != source lab_results count")

print("[STEP 6 COMPLETE] Validation passed")

print("[STEP 7] Computing processed record count")
record_count = hospital_daily_summary_df.count()
print(f"[STEP 7 COMPLETE] Processed record count: {record_count}")
dbutils.notebook.exit(str(record_count))